In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import anndata as ad
import scanpy as sc
import sys
sys.path.append('/home/jwinchel/Heimdall/')

from Heimdall.utils import convert_to_ensembl_ids

/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:

## Format NSCLC scRNAseq Data into AnnData

In [2]:
import os, re

# get human scRNA-seq files
data_dir = '/compute/oven-0-13/jwinchel/scRNA-seq/'
sc_files = sorted([os.path.join(data_dir,x) for x in os.listdir(data_dir) if "human" in x and "human_cell" not in x])

In [3]:
# read in common genes
gene_overlap = pd.read_csv('/compute/oven-0-13/jwinchel/gene_overlap.csv')
common_genes = list(sorted(gene_overlap['ensembl_gene'].unique()))

In [4]:
# read in all scRNA-seq files
adata_list = []
for i,f in enumerate(sc_files):
    
    f_adata = sc.read_text(f, delimiter='\t')
    f_adata,_ = convert_to_ensembl_ids(f_adata,'/compute/oven-0-13/jwinchel/')
    f_adata = f_adata[:,common_genes]
    print(f_adata.shape)
    fname = f.split('/')[-1]
    fname_fields = fname.split('_')
    f_adata.obs['species'] = fname_fields[1]
    f_adata.obs['filename'] = fname.split('.')[0]
    f_adata.obs['sample'] = fname_fields[2]
    f_adata.obs['GSM_ID'] = fname_fields[0]
    adata_list.append(f_adata)

adata = ad.concat(adata_list,join='outer')

KeyboardInterrupt: 

In [31]:
adata.obs['cell_ID_sample'] = adata.obs.index + '_' + adata.obs['sample']
adata.obs['cell_ID'] = adata.obs.index
adata.obs.set_index('cell_ID_sample',inplace=True)

In [45]:
adata.obs

,species,filename,sample,GSM_ID,cell_ID
cell_ID_sample,,,,,
bcIIOD_p1t1,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcIIOD
bcHTNA_p1t1,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcHTNA
bcDLAV_p1t1,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcDLAV
bcHNVA_p1t1,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcHNVA
bcALZN_p1t1,human,GSM3635278_human_p1t1_raw_counts,p1t1,GSM3635278,bcALZN
...,...,...,...,...,...
bcEDIZ_p7b1,human,GSM3635303_human_p7b1_raw_counts,p7b1,GSM3635303,bcEDIZ
bcFTLK_p7b1,human,GSM3635303_human_p7b1_raw_counts,p7b1,GSM3635303,bcFTLK
bcGIVY_p7b1,human,GSM3635303_human_p7b1_raw_counts,p7b1,GSM3635303,bcGIVY


In [3]:
import pandas as pd
# read in cell metadata
metadata = pd.read_csv('/compute/oven-0-13/jwinchel/scRNA-seq/GSE127465_human_cell_metadata_54773x25.tsv',delimiter='\t',index_col='Barcode')
metadata['Barcode'] = metadata.index +'_'+ metadata['Library']
metadata.reset_index(drop=True,inplace=True)
metadata
# print(metadata['Library'])

# adata.obs = pd.merge(adata.obs,metadata,how='left',left_on='cell_ID_sample',right_on='Barcode')


,Patient,Tissue,Barcoding emulsion,Library,Total counts,Percent counts from mitochondrial genes,Most likely LM22 cell type,Major cell type,Minor subset,used_in_NSCLC_all_cells,...,used_in_NSCLC_immune,x_NSCLC_immune,y_NSCLC_immune,used_in_NSCLC_non_immune,x_NSCLC_non_immune,y_NSCLC_non_immune,used_in_blood,x_blood,y_blood,Barcode
0,p1,blood,p1b,p1b1,25465.0,0.058904,T cells CD4 memory activated,bT cells,bCD4T1,False,...,False,NaN,NaN,False,NaN,NaN,True,1620.527869,-205.677730,bcCSBJ_p1b1
1,p1,blood,p1b,p1b1,7582.0,9.206014,Monocytes,bMonocytes,bMono2,False,...,False,NaN,NaN,False,NaN,NaN,True,1688.259525,-965.414268,bcAVUP_p1b1
2,p1,blood,p1b,p1b1,5022.0,5.575468,Monocytes,bpDC,bpDC,False,...,False,NaN,NaN,False,NaN,NaN,True,1537.052487,-719.481146,bcACUH_p1b1
3,p1,blood,p1b,p1b1,3694.0,6.361668,T cells CD4 memory resting,bT cells,bCD4T1,False,...,False,NaN,NaN,False,NaN,NaN,True,1590.280870,-200.568512,bcDLRE_p1b1
4,p1,blood,p1b,p1b1,2945.0,6.213922,NK cells resting,bNK cells,bNK cells,False,...,False,NaN,NaN,False,NaN,NaN,True,1344.771262,228.322604,bcESSY_p1b1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54768,p7,tumor,p7t,p7t2,550.0,18.363636,Eosinophils,tMast cells,tMast1,True,...,True,1874.365630,-157.836426,False,NaN,NaN,False,NaN,NaN,bcFAUI_p7t2
54769,p7,tumor,p7t,p7t2,549.0,5.828780,Macrophages M0,tMoMacDC,tMac1,True,...,True,1275.199283,-1422.543132,False,NaN,NaN,False,NaN,NaN,bcFHTY_p7t2
54770,p7,tumor,p7t,p7t2,429.0,6.060606,Macrophages M2,tMoMacDC,tMac3,True,...,True,1642.186657,-1307.642606,False,NaN,NaN,False,NaN,NaN,bcBHQQ_p7t2
54771,p7,tumor,p7t,p7t2,364.0,5.769231,Neutrophils,tMoMacDC,tMono3,True,...,True,864.411359,-1577.576741,False,NaN,NaN,False,NaN,NaN,bcCXIB_p7t2


In [56]:
adata.obs = adata.obs.astype(str)

In [ ]:
# write anndata with cell type info (with and w/out unlabeled cells)
adata.write_h5ad('/work/magroup/skrieger/Syndri/data/nsclc_scRNAseq_with_unlabeled.h5ad')
adata[~adata.obs['Major cell type'].isin(['nan'])].write_h5ad('/work/magroup/skrieger/Syndri/data/nsclc_scRNAseq.h5ad')